# 🤖 Step 3: Model Training (1D CNN)

Build and train the 1D Convolutional Neural Network.

## What This Notebook Does:
-✅ Load training data from CSV (~100 ground truth samples)
- ✅ Build 1D CNN architecture
- ✅ Train the model (longer training for small dataset)
- ✅ Evaluate performance
- ✅ Save trained model

---

**Previous:** [02_preprocessing.ipynb](02_preprocessing.ipynb)  
**Next:** [04_prediction_analysis.ipynb](04_prediction_analysis.ipynb)

**Note:** Training on small ground truth dataset (~100 samples). Using more epochs and regularization.

## Load Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dense, Flatten, MaxPooling1D, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

# Load training data
print("📂 Loading training data...")
df = pd.read_csv("outputs/training_data.csv")

print(f"✅ Data loaded!")
print(f"   Shape: {df.shape}")
print(f"   Features: B02, B03, B04, B08, NDVI")
print(f"   Target: Label (3 classes from Ground Truth)")
print(f"\nClass distribution:")
class_counts = df['Label'].value_counts().sort_index()
class_names = {1: 'Seagrass', 2: 'Water', 3: 'Sand'}
for label, count in class_counts.items():
    print(f"   Class {label} ({class_names[label]}): {count} samples")

## Prepare Data for CNN

In [ ]:
# Prepare features and labels
X = df[["B02", "B03", "B04", "B08", "NDVI"]].values
y = df["Label"].values

# Convert labels to 0-based indexing for Keras (1,2,3 → 0,1,2)
y = y - 1

# Reshape for 1D CNN (samples, features, channels)
X = X.reshape(-1, 5, 1)

# Split data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set:   {X_train.shape[0]:,} samples")
print(f"Test set:       {X_test.shape[0]:,} samples")
print(f"Input shape:    {X_train.shape[1:]}")
print(f"Output classes: 3 (Seagrass, Water, Sand)")
print(f"\n⚠️  Note: Small dataset! Using longer training + regularization")

## Build 1D CNN Model

**Architecture:**
- Conv1D layer (32 filters, kernel=2)
- MaxPooling1D
- Flatten
- Dense layers with Dropout (increased for regularization)
- Output: 3 classes (softmax)

**Optimized for small dataset:**
- Higher dropout rate (0.5)
- More regularization
- Longer training (50 epochs with early stopping)

In [ ]:
# Build model (optimized for small dataset)
model = Sequential([
    Conv1D(32, kernel_size=2, activation='relu', input_shape=(5, 1)),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),  # Increased from 0.3 for small dataset
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')  # Changed from 4 to 3 classes
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("📊 Model Architecture:")
model.summary()

## Train the Model

In [ ]:
# Train model (longer training for small dataset)
print("🚀 Training model...")
print("⏱️  Using 50 epochs with early stopping (patience=10)")

# Early stopping to prevent overfitting
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    epochs=50,  # Increased from 10
    batch_size=16,  # Smaller batch size for small dataset
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\n✅ Training Complete!")
print(f"   Test Accuracy: {test_accuracy*100:.2f}%")
print(f"   Test Loss: {test_loss:.4f}")
print(f"   Epochs trained: {len(history.history['loss'])}")

# Plot training history
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Model Loss', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Model Accuracy', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/03_training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💾 Training history saved to 'outputs/03_training_history.png'")

## Save the Model

In [ ]:
# Save the trained model
model.save('outputs/coastal_cnn_model.h5')
print("💾 Model saved to 'outputs/coastal_cnn_model.h5'")

# Save metadata
import pickle
model_metadata = {
    'test_accuracy': test_accuracy,
    'test_loss': test_loss,
    'input_shape': (5, 1),
    'output_classes': 3,  # Changed from 4 to 3
    'class_names': {0: 'Seagrass', 1: 'Water', 2: 'Sand'},  # 0-based for predictions
    'class_names_original': {1: 'Seagrass', 2: 'Water', 3: 'Sand'},  # Original labels
    'training_samples': len(X_train),
    'epochs_trained': len(history.history['loss'])
}

with open('outputs/model_metadata.pkl', 'wb') as f:
    pickle.dump(model_metadata, f)

print("✅ Metadata saved to 'outputs/model_metadata.pkl'")
print("\n" + "="*60)
print("✅ MODEL TRAINING COMPLETE!")
print("="*60)
print(f"\nFinal Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Training samples: {len(X_train)}")
print(f"Epochs trained: {len(history.history['loss'])}")
print("\n📌 Next Step: Open 04_prediction_analysis.ipynb to make predictions")